# Super basic recommendation system (V2)

This notebook implements a **hybrid recommendation model** combining **categorical metadata** (genres, themes, demographics) and **semantic narrative similarity** (synopsis text matching) on the Kaggle anime database.

It also supports generating recommendations based on a user's imported list of anime, mimicking the output of a Collaborative Filtering model.

### Features:
1. **Categorical Features**: Multi-hot encodes genres, themes, and demographics.
2. **Semantic Features**: Uses **Sentence Transformers** (`all-MiniLM-L6-v2`) to embed synopsis text, with TF-IDF fallback.
3. **Hybrid Similarity**: Merges categorical and semantic similarity with a adjustable weight parameter $\alpha$.
4. **User Profile Recommender**: Averages the similarity vectors of a user's anime list and ranks candidates.

In [ ]:
try:
    import sentence_transformers
    print("sentence-transformers is already installed.")
except ImportError:
    print("sentence-transformers not found. Installing...")
    !pip install -q sentence-transformers

import os
import ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

### Data Loading & Preprocessing

In [ ]:
ALPHA = 0.5 # Categorical similarity weight so semantic will be 1 - alpha.
USE_SENTENCE_TRANSFORMERS = True  # False = TF-IDF (for CPU)

DATA_PATH = "/kaggle/input/datasets/neelagiriaditya/anime-dataset-jan-1917-to-oct-2025/datasets/details.csv"

print(f"Loading data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
print(f"Successfully loaded {len(df)} anime entries.")

def parse_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('[') and x.endswith(']'):
            try:
                return ast.literal_eval(x)
            except:
                pass
        return [i.strip() for i in x.split(',') if i.strip()]
    return []

df['genres'] = df['genres'].apply(parse_list)
df['themes'] = df['themes'].apply(parse_list)
df['demographics'] = df['demographics'].apply(parse_list)

df['synopsis'] = df['synopsis'].fillna("")
df['title'] = df['title'].fillna("Unknown Title")
df['members'] = pd.to_numeric(df['members'], errors='coerce').fillna(0).astype(int)
df['mal_id'] = pd.to_numeric(df['mal_id'], errors='coerce').fillna(0).astype(int)

print("Sample record preview:")
display(df[['mal_id', 'title', 'genres', 'themes', 'demographics', 'members']].head())

### Categorical Feature Matrix

In [ ]:
def combine_tags(row):
    tags = set()
    tags.update(row['genres'])
    tags.update(row['themes'])
    tags.update(row['demographics'])
    return list(tags)

df['meta_tags'] = df.apply(combine_tags, axis=1)

mlb = MultiLabelBinarizer(sparse_output=True)
categorical_matrix = mlb.fit_transform(df['meta_tags'])

print(f"Total unique tags: {len(mlb.classes_)}")
print("Computing categorical cosine similarity...")
cat_sim = cosine_similarity(categorical_matrix, dense_output=True)
print("Categorical similarity computation complete.")

### Semantic Feature Matrix

In [ ]:
sem_sim = None

if USE_SENTENCE_TRANSFORMERS:
    try:
        from sentence_transformers import SentenceTransformer
        import torch
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading sentence-transformer model ('all-MiniLM-L6-v2') on {device.upper()}...")
        model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
        
        synopses = df['synopsis'].tolist()
        
        print("Encoding synopses (this may take a few minutes depending on hardware)...")
        embeddings = model.encode(synopses, show_progress_bar=True, batch_size=128)
        
        print("Computing semantic cosine similarity...")
        sem_sim = cosine_similarity(embeddings)
        
    except Exception as e:
        print(f"Sentence Transformers failed/not available: {e}")
        print("Switching fallback to TF-IDF...")
        USE_SENTENCE_TRANSFORMERS = False

if not USE_SENTENCE_TRANSFORMERS:
    print("Computing TF-IDF matrices on synopses...")
    tfidf = TfidfVectorizer(max_features=15000, stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df['synopsis'])
    
    print("Computing semantic cosine similarity (TF-IDF)...")
    sem_sim = cosine_similarity(tfidf_matrix, dense_output=True)

print("Semantic similarity computation complete.")

### Compute Hybrid Similarity Matrix

In [ ]:
print(f"Combining similarities (Alpha: {ALPHA} Categorical / {1.0 - ALPHA} Semantic)...")
hybrid_sim = ALPHA * cat_sim + (1 - ALPHA) * sem_sim

np.fill_diagonal(hybrid_sim, 0.0)
print("Hybrid similarity matrix constructed.")

### Generating User Profile Recommendations
We generate recommendations by evaluating the mean similarity to a user's known anime list.

In [ ]:
def get_recommendations(user_anime_list, top_k=20, popularity_weight=0.15):
    """
    Recommend anime given a list of `mal_id`s that the user has watched/liked.
    """
    # Find the indices of the user's anime
    valid_indices = df[df['mal_id'].isin(user_anime_list)].index.tolist()
    
    if not valid_indices:
        print("No valid anime found from the user's list.")
        return [], []
        
    # Get the average similarity profile for the user
    user_profile_sim = hybrid_sim[valid_indices].mean(axis=0)
    
    # Compute log-scaled popularity score (0.0 to 1.0)
    log_members = np.log1p(df['members'].values)
    max_log_members = log_members.max() if log_members.max() > 0 else 1.0
    popularity_scores = log_members / max_log_members
    
    # Combine similarity and popularity
    combined_scores = (1.0 - popularity_weight) * user_profile_sim + popularity_weight * popularity_scores
    
    # Mask out items already in the user's list
    masked_scores = combined_scores.copy()
    masked_scores[valid_indices] = -np.inf
    
    # Get top K
    top_indices = np.argsort(masked_scores)[-top_k:][::-1]
    top_scores = masked_scores[top_indices]
    
    # Map indices back to mal_id
    anime_ids = df.iloc[top_indices]['mal_id'].values
    
    return anime_ids, top_scores

# Mock user list: e.g., Naruto (20) and Bleach (269)
sample_user_list = [20, 269]

print("Sample Recommendations for User Profile:")
recs, scores = get_recommendations(sample_user_list)
for rank, (aid, score) in enumerate(zip(recs, scores), 1):
    print(f"{rank}. Anime ID: {aid} (Score: {score:.3f})")